# Legal Knowledge Graph - Relationship Mining
Quy trình tự động khai phá các quan hệ pháp lý từ tập dữ liệu Hugging Face.
- **Pha 1**: Join `metadata` và `content` qua `id`, dùng Regex quét số hiệu văn bản (Dest Doc Num).
- **Pha 2**: Dùng LLM (Llama-3 8B On-Premise) trích xuất tự do các cụm từ chỉ mối quan hệ (Open Information Extraction).
- **Pha 3**: Thống kê, in ra danh sách các mối quan hệ (chuẩn bị cho bước Gom cụm).
- **Pha 4**: Đề xuất code tích hợp vào FSM Chunker.

In [1]:
# ==========================================
# BƯỚC 0: KHỞI TẠO & IMPORT
# ==========================================
import sys
import os
import re
import json
import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm
from IPython.display import display

# 1. Cấu hình Cache HuggingFace lên ổ D để tránh tràn ổ C
os.environ["HF_HOME"] = r"D:\huggingface_cache"
os.environ["HF_DATASETS_CACHE"] = r"D:\huggingface_cache"
os.environ["TRANSFORMERS_CACHE"] = r"D:\huggingface_cache"
os.environ["SENTENCE_TRANSFORMERS_HOME"] = r"D:\huggingface_cache\sentence_transformers"

# 2. Trỏ vào thư mục gốc Legal-RAG để import LLM Client
sys.path.append(os.path.abspath("d:/iCOMM/Legal-RAG"))

try:
    from backend.llm.client import InternalLLMClient
    llm_client = InternalLLMClient()
    print("✅ Khởi tạo LLM Client nội bộ thành công.")
except ImportError:
    print("⚠️ Không tìm thấy InternalLLMClient.")

# 3. Cấu hình chung
DATASET_NAME = "nhn309261/vietnamese-legal-docs"
MAX_DOCS_TO_SCAN = 1000  # Đặt None nếu muốn quét toàn bộ
IGNORE_KEYWORDS = ["sửa đổi", "bãi bỏ", "thay thế", "căn cứ"]
REGEX_DOC_NUM = r"\b\d+/(?:20\d{2}|19\d{2})/[A-ZĐ][A-Z0-9Đa-zđ\-]*\b|\b\d+/[A-ZĐ][A-Z0-9Đa-zđ\-]*\b"
REGEX_LAW_NAME = r"(?:Hiến pháp|Bộ luật|Luật)\s+[A-ZĐ][a-zđA-ZĐ\s]*(?:năm\s+\d{4})?|(?:Hiến pháp)(?:\s+năm\s+\d{4})?"
REGEX_PATTERN = re.compile(f"({REGEX_DOC_NUM}|{REGEX_LAW_NAME})")
print(f"Regex pattern: {REGEX_PATTERN}")
# Quick test
test = "Nghị định số 100/2019/NĐ-CP và Luật 45/2019/QH14"
print(f"Test matches: {re.findall(REGEX_PATTERN, test)}")

✅ Khởi tạo LLM Client nội bộ thành công.
Regex pattern: re.compile('(\\b\\d+/(?:20\\d{2}|19\\d{2})/[A-ZĐ][A-Z0-9Đa-zđ\\-]*\\b|\\b\\d+/[A-ZĐ][A-Z0-9Đa-zđ\\-]*\\b|(?:Hiến pháp|Bộ luật|Luật)\\s+[A-ZĐ][a-zđA-ZĐ\\s]*(?:năm\\s+\\d{4})?|(?:Hiến pháp)(?:\\s+năm\\s+\\d{4})?)')
Test matches: ['100/2019/NĐ-CP', '45/2019/QH14']


In [2]:
import os
import re
import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm

DATASET_NAME = "nhn309261/vietnamese-legal-docs"
MAX_DOCS_TO_SCAN = 1000  # None = full
RAW_OUTPUT_FILE = "raw_relationships.csv"


def extract_raw_relationships(regex_pattern: str):
    """
    Phase 1: Scan dataset và extract raw relationships
    """
    print("⏳ Đang tải subset 'metadata'...")
    ds_meta = load_dataset(
        DATASET_NAME, "metadata", split="data",
        cache_dir=os.environ.get("HF_DATASETS_CACHE")
    )

    meta_lookup = {
        str(row["id"]): row.get("document_number", "")
        for row in ds_meta
    }
    print(f"✅ Metadata: {len(meta_lookup)} docs")

    print("\n⏳ Đang tải subset 'content'...")
    ds_content = load_dataset(
        DATASET_NAME, "content", split="data",
        cache_dir=os.environ.get("HF_DATASETS_CACHE")
    )

    raw_relationships = []
    docs_to_process = MAX_DOCS_TO_SCAN if MAX_DOCS_TO_SCAN else len(ds_content)

    print(f"🚀 Quét {docs_to_process} documents...")

    for i, row in tqdm(enumerate(ds_content), total=docs_to_process):
        if MAX_DOCS_TO_SCAN and i >= MAX_DOCS_TO_SCAN:
            break

        doc_id = str(row["id"])
        text = row.get("content", "")
        if not text:
            continue

        source_doc_num = meta_lookup.get(doc_id, "") or f"UNKNOWN_{doc_id}"

        paragraphs = [
            p.strip() for p in text.split('\n')
            if len(p.strip()) > 30
        ]

        for para in paragraphs:
            matches = list(re.finditer(regex_pattern, para))
            if not matches:
                continue

            for match in matches:
                dest_doc_num = match.group()

                if dest_doc_num == source_doc_num:
                    continue

                raw_relationships.append({
                    "source_doc_num": source_doc_num,
                    "dest_doc_num": dest_doc_num,
                    "context": para
                })

    df = pd.DataFrame(raw_relationships)

    if df.empty:
        print("⚠️ Không tìm thấy quan hệ nào.")
        return None

    df = df.drop_duplicates(
        subset=["source_doc_num", "dest_doc_num", "context"]
    )

    df.to_csv(RAW_OUTPUT_FILE, index=False, encoding="utf-8")
    print(f"✅ Saved {len(df)} relationships → {RAW_OUTPUT_FILE}")

    return df


def load_or_create_raw(regex_pattern: str):
    """
    Wrapper: nếu CSV đã tồn tại → load, không thì tạo mới
    """
    if os.path.exists(RAW_OUTPUT_FILE):
        print(f"⚡ Found existing file: {RAW_OUTPUT_FILE} → skip extraction")
        return pd.read_csv(RAW_OUTPUT_FILE)

    print("🚀 Chưa có file → chạy extraction...")
    return extract_raw_relationships(regex_pattern)


# =========================
# RUN PIPELINE
# =========================

df_raw = load_or_create_raw(REGEX_PATTERN)

# =========================
# BƯỚC 2 (ceil tiếp theo)
# =========================
if df_raw is not None:
    print("➡️ Chuyển sang bước tiếp theo...")
    # gọi hàm phase 2 ở đây
    # process_with_llm(df_raw)

⚡ Found existing file: raw_relationships.csv → skip extraction
➡️ Chuyển sang bước tiếp theo...


In [3]:
import re

def is_valid_docnum(doc_str):
    """Hàm lọc rác: Kiểm tra xem chuỗi có thực sự là Số hiệu/Tên văn bản pháp lý không"""
    doc_str = str(doc_str).strip()
    doc_lower = doc_str.lower()
    
    # 1. Quá ngắn hoặc bị cụt (chống Luật Đ, Luật Xu...)
    if len(doc_str) < 6: 
        return False
    if doc_str.startswith("Luật ") and len(doc_str.split()) < 3 and not any(char.isdigit() for char in doc_str):
        # Từ chối các Luật có đúng 2 chữ mà không có số hiệu đi kèm (như "Luật H", "Luật K")
        # Ngoại lệ hợp lệ: "Luật Đất đai" (3 chữ)
        if len(doc_str.split()) < 2 or len(doc_str) < 10:
            return False

    # 2. Chứa tên Cơ quan / Tổ chức / Doanh nghiệp
    blocklist_orgs = ["bộ ", "sở ", "tổng công ty", "tập đoàn", "chính phủ", "doanh nghiệp", "ủy ban", "thanh tra"]
    if any(org in doc_lower for org in blocklist_orgs):
        return False

    # 3. Chứa Danh từ chung / Khái niệm
    blocklist_concepts = ["phụ lục", "thủ tục hành chính", "một số thông tư", "giấy phép", "mục", "chương", "danh mục"]
    if any(doc_lower.startswith(concept) for concept in blocklist_concepts):
        return False
        
    return True

In [4]:
# ==========================================
# CẬP NHẬT PHA 2: TRÍCH XUẤT BỘ BA QUAN HỆ (ĐÃ FIX LOGIC)
# ==========================================
import json
import re
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display

extracted_relations = []
print("🚀 Đang dùng LLM 8B để trích xuất Mạng lưới Bộ ba (Triplet)...")

# Chạy thử 200 dòng (Bạn đổi thành df_raw khi muốn chạy toàn bộ)
df_sample = df_raw.head(200)

# 💡 Khai báo danh sách các từ bị cấm xuất hiện trong relation_phrase
FORBIDDEN_ENTITY_WORDS = ["luật", "nghị định", "thông tư", "quyết định", "chỉ thị", "nghị quyết", "điều", "khoản", "hiến pháp", "số"]

for idx, row in tqdm(df_sample.iterrows(), total=len(df_sample)):
    context = row['context']
    source_doc = row['source_doc_num']
    dest_doc = row['dest_doc_num'] 

    # 💡 NÂNG CẤP PROMPT: Thêm Few-shot examples và Blacklist
    prompt = f"""Bạn là Chuyên gia Khai phá Dữ liệu Pháp lý và Kỹ sư Knowledge Graph.
Bạn đang phân tích VĂN BẢN HIỆN TẠI (Source Doc) có số hiệu là: [{source_doc}]

Hãy đọc đoạn văn sau:
"{context}"

NHIỆM VỤ: Trích xuất TẤT CẢ các mối quan hệ pháp lý dưới dạng JSON Bộ ba (Triplets) và tự động phân loại vào các NHÃN ĐỒ THỊ chuẩn hóa.

QUY TẮC BẮT BUỘC:
1. QUAN HỆ BẮC CẦU: "Căn cứ Luật A được sửa đổi bởi Luật B" -> 2 quan hệ: ([{source_doc}] -> căn cứ -> Luật A) VÀ (Luật B -> sửa đổi -> Luật A).
2. ĐẢO CHIỀU BỊ ĐỘNG: Nếu gặp câu mang nghĩa bị động (ví dụ: "Luật A được sửa đổi/thay thế bởi Luật B"), BẮT BUỘC đảo lại chiều tác động. Nguồn (Source) phải là kẻ thực thi: [Luật B]. Đích (Target) là kẻ bị tác động: [Luật A].
3. LUẬT ĐA HỢP: "Luật Sửa đổi, bổ sung một số điều của Luật A số X/Năm" là MỘT thực thể duy nhất. TUYỆT ĐỐI KHÔNG tách chữ "sửa đổi" trong tên văn bản này thành quan hệ.
4. KHUYẾT CHỦ NGỮ: Câu bắt đầu bằng "Căn cứ...", "Thực hiện..." thì Source tự động là [{source_doc}].
5. BỊ BÃI BỎ/HẾT HIỆU LỰC: "Luật B bị bãi bỏ" hoặc "hết hiệu lực" -> Kẻ tước hiệu lực là văn bản hiện tại. Source: [{source_doc}], Target: [Luật B].
6. GIỚI HẠN CỤM TỪ (RELATION_PHRASE) - RẤT QUAN TRỌNG:
   - CHỈ lấy động từ/giới từ cốt lõi (Tối đa 5 từ).
   - DANH SÁCH CẤM: TUYỆT ĐỐI KHÔNG chứa các từ chỉ loại văn bản (Luật, Nghị định, Quyết định, Chỉ thị, Hiến pháp) hay số hiệu trong trường này.
   - VÍ DỤ ĐÚNG: "căn cứ", "sửa đổi, bổ sung", "bãi bỏ", "thực hiện theo".
7. GÁN NHÃN ĐỒ THỊ (EDGE_LABEL): Bắt buộc phân loại "relation_phrase" vào MỘT TRONG 10 nhãn chuẩn dưới đây dựa trên bản chất ngữ nghĩa:
   - BASED_ON (Nền tảng ban hành/Thẩm định): Văn bản này lấy văn bản kia làm cơ sở pháp lý để ra đời. 
     + Cụm từ thường gặp: căn cứ, căn cứ quy định tại, xét, theo đề nghị, theo ý kiến thẩm định...
   - AMENDS (Sửa đổi nội dung): Thay đổi, thêm hoặc bớt một phần nội dung văn bản (văn bản gốc vẫn còn hiệu lực).
     + Cụm từ thường gặp: sửa đổi, bổ sung, cắt giảm, cắt giảm theo...
   - REPEALS (Chấm dứt hiệu lực): Khai tử, xóa bỏ hoàn toàn giá trị pháp lý của một văn bản.
     + Cụm từ thường gặp: bãi bỏ, hết hiệu lực, hết hiệu lực thi hành...
   - REPLACES (Thay thế toàn diện): Bỏ hẳn văn bản cũ và dùng văn bản hiện tại để thay thế.
     + Cụm từ thường gặp: thay thế...
   - GUIDES (Chi tiết hóa, Hướng dẫn): Làm rõ cách thức thực hiện, cụ thể hóa cho một văn bản cấp cao hơn.
     + Cụm từ thường gặp: hướng dẫn thi hành, quy định chi tiết, hướng dẫn tại...
   - APPLIES (Tuân thủ, Hành động thực thi): Đưa quy định vào thực tiễn, làm theo chỉ đạo hoặc lộ trình đã vạch ra.
     + Cụm từ thường gặp: thực hiện, thực hiện theo, áp dụng, tổ chức hoạt động theo, hoàn thành theo lộ trình, bảo đảm lộ trình theo...
   - REFERENCES (Dẫn chiếu, Viện dẫn): Nhắc đến nội dung, định nghĩa hoặc chính sách ở văn bản khác để làm rõ ngữ cảnh (không mang tính ra lệnh thực thi trực tiếp).
     + Cụm từ thường gặp: quy định tại, theo quy định tại, liên quan đến, nghiên cứu, hưởng chính sách tại...
   - ISSUED_WITH (Đính kèm cấu trúc): Văn bản phụ (Phụ lục, biểu mẫu, quy chế) đi kèm vật lý với văn bản chính.
     + Cụm từ thường gặp: ban hành kèm theo, theo mẫu ban hành kèm theo...
   - ASSIGNS (Phân công, Trách nhiệm): Giao việc, chỉ định thẩm quyền cho tổ chức/cá nhân.
     + Cụm từ thường gặp: giao nhiệm vụ tại, chịu trách nhiệm về...
   - CORRECTS (Sửa lỗi kỹ thuật): Chữa lỗi chính tả, sai sót kỹ thuật văn bản.
     + Cụm từ thường gặp: đính chính...
8. CHUẨN HÓA TÊN VĂN BẢN (SOURCE/TARGET) - CỰC KỲ QUAN TRỌNG:
   - Source và Target BẮT BUỘC phải là Số hiệu văn bản (ví dụ: 123/2024/NĐ-CP, 183-KL/TW) HOẶC Tên đầy đủ của Luật/Bộ luật/Hiến pháp (ví dụ: Luật Đất đai, Bộ luật Dân sự, Hiến pháp).
   - CHỐNG CỤT ĐUÔI: TUYỆT ĐỐI KHÔNG viết tắt, không cắt cụt tên Luật. (SAI: "Luật Đ", "Luật Ban H", "Luật Xu" -> ĐÚNG: Phải ghi đủ "Luật Đất đai", "Luật Ban hành văn bản quy phạm pháp luật").
   - CHỐNG NHIỄU: TUYỆT ĐỐI KHÔNG bốc tên Cơ quan/Tổ chức (ví dụ: Bộ Xây dựng, Bộ Tài chính, Chính phủ), KHÔNG bốc danh từ chung (ví dụ: Phụ lục 01, thủ tục hành chính, một số Thông tư, Giấy phép) để làm Source hoặc Target. Nếu không tìm thấy văn bản đích thực sự, hãy bỏ qua.

VÍ DỤ ĐẦU RA CHUẨN (FEW-SHOT):
{{
  "relationships": [
    {{
      "source": "{source_doc}",
      "target": "Luật Tổ chức Chính phủ",
      "relation_phrase": "căn cứ",
      "edge_label": "BASED_ON"
    }},
    {{
      "source": "Nghị định số 123/2016/NĐ-CP",
      "target": "Nghị định 101/2020/NĐ-CP",
      "relation_phrase": "sửa đổi",
      "edge_label": "AMENDS"
    }}
  ]
}}

CHỈ TRẢ VỀ ĐÚNG ĐỊNH DẠNG JSON MẢNG, KHÔNG GIẢI THÍCH MỘT TỪ NÀO KHÁC.
"""

    try:
        resp = llm_client.chat_completion(
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
            response_format={"type": "json_object"}
        )
        
        resp = resp or ""
        match = re.search(r'\{.*\}', resp, re.DOTALL)
        
        if match:
            data = json.loads(match.group(0))
            for rel in data.get("relationships", []):
                s = rel.get("source", "").strip()
                t = rel.get("target", "").strip()
                p = rel.get("relation_phrase", "").strip().lower()
                l = rel.get("edge_label", "UNKNOWN").strip().upper()
                
                # 🛑 ĐƯA HÀM LỌC VÀO ĐÂY: Chặn đứng rác trước khi đưa vào list
                if not s or not t or s == t:
                    continue
                if not is_valid_docnum(s) or not is_valid_docnum(t):
                    # print(f"Đã chặn rác: {s} -> {t}") # Bỏ comment nếu muốn xem những gì bị chặn
                    continue
                if re.fullmatch(r'\d{1,2}/\d{1,2}/\d{4}', t) or re.fullmatch(r'\d{1,2}/\d{1,2}', t):
                    continue
                    
                extracted_relations.append({
                    "source_doc": s,
                    "target_doc": t,
                    "relation_phrase": p,
                    "edge_label": l,
                    "context": context
                })
    except Exception as e:
        print(f"⚠️ Lỗi tại dòng {idx}: {e}")
        continue

# 3. Gom và hiển thị kết quả
df_triplets = pd.DataFrame(extracted_relations)

if not df_triplets.empty:
    df_triplets = df_triplets.drop_duplicates(subset=["source_doc", "target_doc", "relation_phrase"])
    print(f"\n✅ Đã trích xuất chuẩn hóa được {len(df_triplets)} Mạng lưới quan hệ (Triple).")
    
    print("\n📄 CHI TIẾT MỘT SỐ QUAN HỆ SAU KHI FIX LOGIC:")
    display(df_triplets.head(10))
    
    # In ra để bạn kiểm tra xem còn bị lọt "luật báo chí", "thông tư số" không
    print("\n✨ CÁC CỤM TỪ QUAN HỆ DUY NHẤT SAU KHI ĐÃ SỬA:")
    print(df_triplets['relation_phrase'].unique())
    
    df_triplets.to_csv("triplet_relations_clean.csv", index=False, encoding="utf-8")
else:
    print("⚠️ Bảng trích xuất rỗng.")

🚀 Đang dùng LLM 8B để trích xuất Mạng lưới Bộ ba (Triplet)...


  0%|          | 0/200 [00:00<?, ?it/s]


✅ Đã trích xuất chuẩn hóa được 150 Mạng lưới quan hệ (Triple).

📄 CHI TIẾT MỘT SỐ QUAN HỆ SAU KHI FIX LOGIC:


,source_doc,target_doc,relation_phrase,edge_label,context
0,115/NQ-HĐBCQG,Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam,căn cứ,BASED_ON,Căn cứ Hiến pháp nước Cộng hòa xã hội chủ nghĩ...
1,Nghị quyết số 203/2025/QH15,Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam,"sửa đổi, bổ sung",AMENDS,Căn cứ Hiến pháp nước Cộng hòa xã hội chủ nghĩ...
2,115/NQ-HĐBCQG,Luật Bầu cử đại biểu Quốc hội và đại biểu Hội ...,căn cứ,BASED_ON,Căn cứ Luật Bầu cử đại biểu Quốc hội và đại bi...
3,Luật số 83/2025/QH15,Luật Bầu cử đại biểu Quốc hội và đại biểu Hội ...,"sửa đổi, bổ sung",AMENDS,Căn cứ Luật Bầu cử đại biểu Quốc hội và đại bi...
6,34/2025/QĐ-UBND,Luật Tổ chức chính quyền địa phương,căn cứ,BASED_ON,Căn cứ Luật Tổ chức chính quyền địa phương số ...
7,34/2025/QĐ-UBND,Luật Ban hành văn bản quy phạm pháp luật,căn cứ,BASED_ON,Căn cứ Luật Ban hành văn bản quy phạm pháp luậ...
8,Luật số 87/2025/QH15,Luật Ban hành văn bản quy phạm pháp luật,"sửa đổi, bổ sung",AMENDS,Căn cứ Luật Ban hành văn bản quy phạm pháp luậ...
11,34/2025/QĐ-UBND,Luật Giáo dục,căn cứ,BASED_ON,Căn cứ Luật Giáo dục số 43/2019/QH14;
12,34/2025/QĐ-UBND,Nghị quyết số 76/2025/UBTVQH15,căn cứ,BASED_ON,Căn cứ Nghị quyết số 76/2025/UBTVQH15 ngày 14 ...
13,34/2025/QĐ-UBND,66/2025/NĐ-CP,căn cứ,BASED_ON,Căn cứ Nghị định số 66/2025/NĐ-CP ngày 12 thán...



✨ CÁC CỤM TỪ QUAN HỆ DUY NHẤT SAU KHI ĐÃ SỬA:
<ArrowStringArray>
[                     'căn cứ',            'sửa đổi, bổ sung',
           'theo quy định tại', 'thực hiện theo quy định tại',
       'hết hiệu lực thi hành',        'hưởng chính sách tại',
                'theo đề nghị',       'theo ý kiến thẩm định',
                         'xét',              'thực hiện theo',
                        'theo',                   'thực hiện',
    'hoàn thành theo lộ trình',       'bảo đảm lộ trình theo',
           'giao nhiệm vụ tại',                    'thay thế',
                'quy định tại',               'hướng dẫn tại',
          'hướng dẫn thi hành',                      'bãi bỏ',
         'chịu trách nhiệm về',   'theo mẫu số 1.1 phụ lục i',
  'theo mẫu ban hành kèm theo',           'quy định chi tiết',
  'quy định chi tiết thi hành',           'ban hành kèm theo',
                     'sửa đổi',                  'đính chính']
Length: 28, dtype: str


In [5]:
# ==========================================
# BƯỚC 3: IN DANH SÁCH & PHÂN TÍCH (PHA 3)
# ==========================================
# Lưu ý: Biến đầu ra từ Bước 2 giờ là df_triplets
if not df_triplets.empty:
    # # 1. Thống kê tần suất xuất hiện của các NHÃN ĐỒ THỊ (Edge Labels)
    edge_counts = df_triplets['edge_label'].value_counts().reset_index()
    edge_counts.columns = ['Nhãn Quan hệ (Edge Label)', 'Số lượng']
    
    print("\n🔥 THỐNG KÊ CÁC NHÃN QUAN HỆ CHUẨN HÓA (DÙNG CHO NEO4J):")
    display(edge_counts)

    # 2. Thống kê cụm từ gốc để đối chiếu xem LLM phân loại có chuẩn không
    phrase_counts = df_triplets['relation_phrase'].value_counts().reset_index()
    phrase_counts.columns = ['Cụm từ gốc (Relation Phrase)', 'Số lượng']
    
    print("\n📝 TOP 15 CỤM TỪ GỐC XUẤT HIỆN NHIỀU NHẤT:")
    display(phrase_counts.head(15))
    
    print("\n🔍 CHI TIẾT 10 BỘ BA (TRIPLET) ĐẦU TIÊN:")
    display(df_triplets.head(10))
    
    # 3. Export ra CSV để nạp thẳng vào Neo4j hoặc kiểm tra thủ công
    output_file = "triplet_relations_clean.csv"
    df_triplets.to_csv(output_file, index=False, encoding="utf-8")
    print(f"\n✅ Đã lưu toàn bộ mạng lưới quan hệ ra file '{output_file}'")
else:
    print("⚠️ Bảng trích xuất trống. Hãy kiểm tra lại cấu hình API LLM ở Bước 2.")


🔥 THỐNG KÊ CÁC NHÃN QUAN HỆ CHUẨN HÓA (DÙNG CHO NEO4J):


,Nhãn Quan hệ (Edge Label),Số lượng
0,BASED_ON,73
1,AMENDS,24
2,APPLIES,22
3,REFERENCES,12
4,ISSUED_WITH,6
5,GUIDES,5
6,REPEALS,4
7,ASSIGNS,2
8,REPLACES,1
9,CORRECTS,1



📝 TOP 15 CỤM TỪ GỐC XUẤT HIỆN NHIỀU NHẤT:


,Cụm từ gốc (Relation Phrase),Số lượng
0,căn cứ,68
1,"sửa đổi, bổ sung",23
2,thực hiện theo,9
3,thực hiện,7
4,theo quy định tại,6
5,ban hành kèm theo,4
6,thực hiện theo quy định tại,3
7,theo,3
8,quy định tại,3
9,bãi bỏ,3



🔍 CHI TIẾT 10 BỘ BA (TRIPLET) ĐẦU TIÊN:


,source_doc,target_doc,relation_phrase,edge_label,context
0,115/NQ-HĐBCQG,Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam,căn cứ,BASED_ON,Căn cứ Hiến pháp nước Cộng hòa xã hội chủ nghĩ...
1,Nghị quyết số 203/2025/QH15,Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam,"sửa đổi, bổ sung",AMENDS,Căn cứ Hiến pháp nước Cộng hòa xã hội chủ nghĩ...
2,115/NQ-HĐBCQG,Luật Bầu cử đại biểu Quốc hội và đại biểu Hội ...,căn cứ,BASED_ON,Căn cứ Luật Bầu cử đại biểu Quốc hội và đại bi...
3,Luật số 83/2025/QH15,Luật Bầu cử đại biểu Quốc hội và đại biểu Hội ...,"sửa đổi, bổ sung",AMENDS,Căn cứ Luật Bầu cử đại biểu Quốc hội và đại bi...
6,34/2025/QĐ-UBND,Luật Tổ chức chính quyền địa phương,căn cứ,BASED_ON,Căn cứ Luật Tổ chức chính quyền địa phương số ...
7,34/2025/QĐ-UBND,Luật Ban hành văn bản quy phạm pháp luật,căn cứ,BASED_ON,Căn cứ Luật Ban hành văn bản quy phạm pháp luậ...
8,Luật số 87/2025/QH15,Luật Ban hành văn bản quy phạm pháp luật,"sửa đổi, bổ sung",AMENDS,Căn cứ Luật Ban hành văn bản quy phạm pháp luậ...
11,34/2025/QĐ-UBND,Luật Giáo dục,căn cứ,BASED_ON,Căn cứ Luật Giáo dục số 43/2019/QH14;
12,34/2025/QĐ-UBND,Nghị quyết số 76/2025/UBTVQH15,căn cứ,BASED_ON,Căn cứ Nghị quyết số 76/2025/UBTVQH15 ngày 14 ...
13,34/2025/QĐ-UBND,66/2025/NĐ-CP,căn cứ,BASED_ON,Căn cứ Nghị định số 66/2025/NĐ-CP ngày 12 thán...



✅ Đã lưu toàn bộ mạng lưới quan hệ ra file 'triplet_relations_clean.csv'


In [12]:
# ==========================================
# BƯỚC HẬU XỬ LÝ: CHUẨN HÓA THỰC THỂ (ENTITY NORMALIZATION)
# Mục tiêu: Ưu tiên lấy Số hiệu văn bản -> Bộ luật/Luật/Hiến pháp
# ==========================================
import pandas as pd
import re
from IPython.display import display

input_file = "triplet_relations_clean.csv"
output_file = "final_kg_triplets.csv"

try:
    df_triplets = pd.read_csv(input_file)
except FileNotFoundError:
    print(f"⚠️ Không tìm thấy file {input_file}. Hãy chạy lại Pha 2.")
    df_triplets = pd.DataFrame()

if not df_triplets.empty:
    print("🚀 Đang tiến hành chuẩn hóa Thực thể (Nodes)...")

# 1. Regex bọc thép Số hiệu và Tên Luật
    REGEX_DOC_NUM = r"\b\d+[\/\-](?:20\d{2}|19\d{2})[\/\-][A-ZĐ][A-Z0-9Đa-zđ\-\/]*\b|\b\d+[\/\-][A-ZĐ][A-Z0-9Đa-zđ\-\/]*\b"
    REGEX_LAW_NAME = r"(?i)(?:Hiến pháp|Bộ luật|Luật)\s+[\w\s\,\.\-]+(?:năm\s+\d{4})?"

    # 🌟 BỔ SUNG: Từ điển thu gọn các Luật Đa Hợp (Omnibus Laws)
    # Bạn có thể bổ sung thêm các luật dài khác vào đây
# 🌟 BỘ TỪ ĐIỂN LIÊN KẾT THỰC THỂ
    OMNIBUS_LAWS_MAPPER = {
        # --- NHÓM LUẬT ĐA HỢP (OMNIBUS LAWS) ---
        "37 luật có liên quan đến quy hoạch": "35/2018/QH14",
        "luật chứng khoán, luật kế toán, luật kiểm toán độc lập": "56/2024/QH15",
        
        # --- NHÓM HIẾN PHÁP & CÁC BỘ LUẬT CHÍNH ---
        "hiến pháp nước cộng hòa xã hội chủ nghĩa việt nam": "Hiến pháp 2013",
        "bộ luật dân sự": "91/2015/QH13",
        "bộ luật hình sự": "100/2015/QH13",
        "bộ luật lao động": "45/2019/QH14",
        "bộ luật tố tụng hình sự": "101/2015/QH13",
        "bộ luật tố tụng dân sự": "92/2015/QH13",
        
        # --- NHÓM LUẬT TỔ CHỨC BỘ MÁY NHÀ NƯỚC ---
        "luật bầu cử đại biểu quốc hội và đại biểu hội đồng nhân dân": "85/2015/QH13",
        "luật tổ chức chính quyền địa phương": "77/2015/QH13",
        "luật tổ chức chính phủ": "76/2015/QH13",
        "luật ban hành văn bản quy phạm pháp luật": "80/2015/QH13",
        "luật thi đua, khen thưởng": "06/2022/QH15",
        
        # --- NHÓM TÀI CHÍNH, THUẾ, KẾ TOÁN ---
        "luật ngân sách nhà nước": "83/2015/QH13",
        "luật quản lý thuế": "38/2019/QH14",
        "luật thuế giá trị gia tăng": "48/2024/QH15",
        "luật thuế thu nhập cá nhân": "04/2007/QH12",
        "luật thuế xuất khẩu, thuế nhập khẩu": "107/2016/QH13",
        "luật kế toán": "88/2015/QH13",
        "luật kiểm toán độc lập": "67/2011/QH12",
        "luật quản lý, sử dụng tài sản công": "15/2017/QH14",
        "luật dự trữ quốc gia": "22/2012/QH13",
        "luật hải quan": "54/2014/QH13",
        
        # --- NHÓM ĐẦU TƯ, KINH DOANH, CHỨNG KHOÁN ---
        "luật đầu tư theo phương thức đối tác công tư": "64/2020/QH14", 
        "luật đầu tư công": "39/2019/QH14", 
        "luật đầu tư": "61/2020/QH14",
        "luật doanh nghiệp": "59/2020/QH14",
        "luật đấu thầu": "22/2023/QH15",
        "luật chứng khoán": "54/2019/QH14",
        "luật các tổ chức tín dụng": "32/2024/QH15",
        
        # --- NHÓM ĐẤT ĐAI, NHÀ Ở, BẤT ĐỘNG SẢN ---
        "luật đất đai": "31/2024/QH15",
        "luật nhà ở": "27/2023/QH15",
        "luật kinh doanh bất động sản": "29/2023/QH15",
        
        # --- NHÓM XÃ HỘI, Y TẾ, GIÁO DỤC, TRUYỀN THÔNG ---
        "luật bảo hiểm xã hội": "41/2024/QH15",
        "luật giáo dục": "43/2019/QH14",
        "luật khám bệnh, chữa bệnh": "15/2023/QH15",
        "luật xử lý vi phạm hành chính": "15/2012/QH13",
        "luật xuất bản": "19/2012/QH13",
        "luật báo chí": "103/2016/QH13",
        "luật an toàn, vệ sinh lao động": "84/2015/QH13",
    }

    # 🚨 ĐẶC BIỆT QUAN TRỌNG: Sắp xếp từ điển theo độ dài tên giảm dần
    SORTED_MAPPER = dict(sorted(OMNIBUS_LAWS_MAPPER.items(), key=lambda item: len(item[0]), reverse=True))

    def normalize_entity(text):
        if pd.isna(text):
            return "UNKNOWN"
        text = str(text).strip()

        # DỌN RÁC TRƯỚC TIÊN
        text = re.sub(r"^(số|của|tại|theo|quy định tại|căn cứ)\s+", "", text, flags=re.IGNORECASE).strip()

        # Ưu tiên 1: Dò tìm Số hiệu văn bản
        num_match = re.search(REGEX_DOC_NUM, text)
        if num_match:
            return num_match.group(0).upper()

        # Ưu tiên 2: Dò tìm Tên Luật
        law_match = re.search(REGEX_LAW_NAME, text)
        if law_match:
            raw_law_name = law_match.group(0).strip()
            
            # 🌟 BƯỚC MỚI: Quét qua từ điển đã được sắp xếp
            lower_law_name = raw_law_name.lower()
            for key_phrase, short_name in SORTED_MAPPER.items():
                # 🚨 ĐÃ XÓA ĐIỀU KIỆN `and "sửa đổi" in lower_law_name`
                if key_phrase in lower_law_name: 
                    return short_name
            
            # Nếu không trúng từ điển thì Format lại cho đẹp như cũ
            return raw_law_name.title().replace("Năm", "năm")

        return text

    # Áp dụng hàm chuẩn hóa cho cột source_doc và target_doc
    df_triplets['source_doc_norm'] = df_triplets['source_doc'].apply(normalize_entity)
    df_triplets['target_doc_norm'] = df_triplets['target_doc'].apply(normalize_entity)

    # Lọc bỏ các dòng lỗi (Source và Target trùng nhau sau khi chuẩn hóa)
    df_triplets = df_triplets[df_triplets['source_doc_norm'] != df_triplets['target_doc_norm']]

    # Tổ chức lại DataFrame cho gọn gàng (Chỉ giữ lại các cột chuẩn)
    df_final = df_triplets[[
        'source_doc_norm', 
        'edge_label', 
        'target_doc_norm', 
        'relation_phrase', 
        'context'
    ]].rename(columns={
        'source_doc_norm': 'source',
        'target_doc_norm': 'target',
        'edge_label': 'relation'
    })

    # Xóa các dòng trùng lặp (nếu có) sau khi đã chuẩn hóa tên
    df_final = df_final.drop_duplicates(subset=['source', 'relation', 'target'])
 
    # Xuất file cuối cùng sẵn sàng cho Neo4j
    df_final.to_csv(output_file, index=False, encoding="utf-8")
    
    print(f"✅ Chuẩn hóa thành công! Đã lưu file cuối cùng: '{output_file}'")
    
    print("\n🔍 KẾT QUẢ SO SÁNH (TRƯỚC VÀ SAU KHI CHUẨN HÓA):")
    demo_df = df_triplets[['source_doc', 'source_doc_norm', 'target_doc', 'target_doc_norm']].head(10)
    display(demo_df)
    
    print("\n📊 BẢNG DỮ LIỆU CUỐI CÙNG SẴN SÀNG NẠP VÀO NEO4J:")
    display(df_final.head(10))

🚀 Đang tiến hành chuẩn hóa Thực thể (Nodes)...
✅ Chuẩn hóa thành công! Đã lưu file cuối cùng: 'final_kg_triplets.csv'

🔍 KẾT QUẢ SO SÁNH (TRƯỚC VÀ SAU KHI CHUẨN HÓA):


,source_doc,source_doc_norm,target_doc,target_doc_norm
0,115/NQ-HĐBCQG,115/NQ-HĐBCQG,Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam,Hiến pháp 2013
1,Nghị quyết số 203/2025/QH15,203/2025/QH15,Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam,Hiến pháp 2013
2,115/NQ-HĐBCQG,115/NQ-HĐBCQG,Luật Bầu cử đại biểu Quốc hội và đại biểu Hội ...,85/2015/QH13
3,Luật số 83/2025/QH15,83/2025/QH15,Luật Bầu cử đại biểu Quốc hội và đại biểu Hội ...,85/2015/QH13
4,34/2025/QĐ-UBND,34/2025/QĐ-UBND,Luật Tổ chức chính quyền địa phương,77/2015/QH13
5,34/2025/QĐ-UBND,34/2025/QĐ-UBND,Luật Ban hành văn bản quy phạm pháp luật,80/2015/QH13
6,Luật số 87/2025/QH15,87/2025/QH15,Luật Ban hành văn bản quy phạm pháp luật,80/2015/QH13
7,34/2025/QĐ-UBND,34/2025/QĐ-UBND,Luật Giáo dục,43/2019/QH14
8,34/2025/QĐ-UBND,34/2025/QĐ-UBND,Nghị quyết số 76/2025/UBTVQH15,76/2025/UBTVQH15
9,34/2025/QĐ-UBND,34/2025/QĐ-UBND,66/2025/NĐ-CP,66/2025/NĐ-CP



📊 BẢNG DỮ LIỆU CUỐI CÙNG SẴN SÀNG NẠP VÀO NEO4J:


,source,relation,target,relation_phrase,context
0,115/NQ-HĐBCQG,BASED_ON,Hiến pháp 2013,căn cứ,Căn cứ Hiến pháp nước Cộng hòa xã hội chủ nghĩ...
1,203/2025/QH15,AMENDS,Hiến pháp 2013,"sửa đổi, bổ sung",Căn cứ Hiến pháp nước Cộng hòa xã hội chủ nghĩ...
2,115/NQ-HĐBCQG,BASED_ON,85/2015/QH13,căn cứ,Căn cứ Luật Bầu cử đại biểu Quốc hội và đại bi...
3,83/2025/QH15,AMENDS,85/2015/QH13,"sửa đổi, bổ sung",Căn cứ Luật Bầu cử đại biểu Quốc hội và đại bi...
4,34/2025/QĐ-UBND,BASED_ON,77/2015/QH13,căn cứ,Căn cứ Luật Tổ chức chính quyền địa phương số ...
5,34/2025/QĐ-UBND,BASED_ON,80/2015/QH13,căn cứ,Căn cứ Luật Ban hành văn bản quy phạm pháp luậ...
6,87/2025/QH15,AMENDS,80/2015/QH13,"sửa đổi, bổ sung",Căn cứ Luật Ban hành văn bản quy phạm pháp luậ...
7,34/2025/QĐ-UBND,BASED_ON,43/2019/QH14,căn cứ,Căn cứ Luật Giáo dục số 43/2019/QH14;
8,34/2025/QĐ-UBND,BASED_ON,76/2025/UBTVQH15,căn cứ,Căn cứ Nghị quyết số 76/2025/UBTVQH15 ngày 14 ...
9,34/2025/QĐ-UBND,BASED_ON,66/2025/NĐ-CP,căn cứ,Căn cứ Nghị định số 66/2025/NĐ-CP ngày 12 thán...
